# Stage 2 — Dental Anomaly Classifier (v4)

**DENTEX-2023 multi-task JSON format:**
- `category_id_1` → quadrant (1–4)
- `category_id_2` → tooth number within quadrant (1–8)
- `category_id_3` → **disease class** (0=healthy, 1–4=disease) ← **this is what we use**
- `categories_3` → disease class name list

We train Faster R-CNN on disease bboxes where `category_id_3 > 0`.
Annotations with `category_id_3 == 0` (healthy) are skipped in training — healthy is inferred by absence of detection.

In [ ]:
# ================== CELL 1: SETUP ==================
import os, json, random, copy
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import nms
from torch.utils.data import Dataset, DataLoader

print('[CELL 1] Starting setup...')
print(f'  PyTorch:     {torch.__version__}')
print(f'  Torchvision: {torchvision.__version__}')
print(f'  CUDA:        {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU:         {torch.cuda.get_device_name(0)}')

STAGE1_WEIGHTS = '/kaggle/input/datasets/ethelrani/maskrcnn-teeth-stage1-weights/maskrcnn_teeth_best.pth'
DENTEX_ROOT    = '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023'
DENTEX_DISEASE_DIR = os.path.join(DENTEX_ROOT, 'training_data/training_data/quadrant-enumeration-disease')
DENTEX_IMG_DIR     = os.path.join(DENTEX_DISEASE_DIR, 'xrays')
DENTEX_ANN_FILE    = os.path.join(DENTEX_DISEASE_DIR, 'train_quadrant_enumeration_disease.json')
INFER_IMG_DIR = os.path.join(DENTEX_ROOT, 'validation_data/validation_data/quadrant_enumeration_disease/xrays')

for label, path in [
    ('Stage1 weights',    STAGE1_WEIGHTS),
    ('DENTEX images',     DENTEX_IMG_DIR),
    ('DENTEX train JSON', DENTEX_ANN_FILE),
    ('Inference xrays',   INFER_IMG_DIR),
]:
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {status}  {label}: {path}')

if not os.path.exists(DENTEX_ANN_FILE):
    print('\n[CELL 1] ⚠  JSON not found — probing disease dir...')
    if os.path.isdir(DENTEX_DISEASE_DIR):
        for f in os.listdir(DENTEX_DISEASE_DIR): print(f'    {f}')
    else:
        print(f'  ⚠  Disease dir not found: {DENTEX_DISEASE_DIR}')

# Placeholder — will be replaced by Cell 2 after parsing categories_3
ANOMALY_CLASSES = {0: 'background'}
NUM_CLASSES2    = 1
ANOMALY_COLORS  = {'Healthy': (0, 220, 100)}

CONFIG2 = {
    'batch_size':     2,
    'lr':             5e-5,
    'epochs':         30,
    'conf_threshold': 0.35,
    'nms_iou':        0.3,
    'train_ratio':    0.75,
    'val_ratio':      0.15,
    'num_workers':    2,
    'patience':       6,
}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n[CELL 1] ✓ Device: {DEVICE}')
print('[CELL 1] ✓ Setup complete.')

In [ ]:
# ================== CELL 2: DATASET (DENTEX MULTI-TASK FORMAT) ==================
# DENTEX uses:  category_id_3  =  disease class (0=healthy, 1..N=disease)
#              categories_3   =  [{"id": 1, "name": "Impacted"}, ...]
# We only train on annotations where category_id_3 > 0 (diseased teeth).
# Healthy inference = no Stage 2 detection on that tooth crop.

with open(DENTEX_ANN_FILE) as f:
    _raw = json.load(f)

print('[CELL 2] JSON top-level keys:', list(_raw.keys()))

# ─ images ────────────────────────────────────────────────────────────────────────────
_images = _raw.get('images', [])
print(f'  images       : {len(_images)} entries')
if _images: print(f'  sample image : {_images[0]}')

# ─ annotations ──────────────────────────────────────────────────────────────────
_anns = _raw.get('annotations', [])
print(f'  annotations  : {len(_anns)} entries')
if _anns: print(f'  sample ann   : {_anns[0]}')

# ─ disease categories from categories_3 ──────────────────────────────────────
_cats3 = _raw.get('categories_3', [])
print(f'  categories_3 : {_cats3}')

_COLOR_POOL = {
    'Impacted':          (255, 100,   0),
    'Caries':            (255,  50,  50),
    'Deep Caries':       (200,   0, 200),
    'Periapical Lesion': (255, 200,   0),
}

if _cats3:
    # Standard case: build class map from categories_3
    # category_id_3 == 0 means healthy (no disease) — skip in training
    # Remap to 1-indexed for Faster R-CNN (background=0, disease=1..N)
    _disease_cats = [c for c in sorted(_cats3, key=lambda c: c['id']) if c['id'] != 0]
    ANOMALY_CLASSES = {0: 'background'}
    # Build a remap: original category_id_3 -> contiguous 1-indexed label
    CAT3_REMAP = {}  # original_id -> model_label
    for new_id, cat in enumerate(_disease_cats, start=1):
        ANOMALY_CLASSES[new_id] = cat['name']
        CAT3_REMAP[cat['id']]   = new_id
    print(f'  ✓ ANOMALY_CLASSES (from categories_3): {ANOMALY_CLASSES}')
    print(f'  ✓ CAT3_REMAP (orig_id -> model_label): {CAT3_REMAP}')
else:
    # Fallback: infer from category_id_3 values in annotations
    _ids3 = sorted(set(a.get('category_id_3', -1) for a in _anns if a.get('category_id_3', 0) > 0))
    print(f'  ⚠  No categories_3 key. Unique non-zero category_id_3: {_ids3}')
    _DENTEX_KNOWN = {1: 'Impacted', 2: 'Caries', 3: 'Deep Caries', 4: 'Periapical Lesion'}
    ANOMALY_CLASSES = {0: 'background'}
    CAT3_REMAP = {}
    for new_id, orig_id in enumerate(_ids3, start=1):
        ANOMALY_CLASSES[new_id] = _DENTEX_KNOWN.get(orig_id, f'disease_{orig_id}')
        CAT3_REMAP[orig_id]     = new_id
    print(f'  ✓ ANOMALY_CLASSES (fallback inferred): {ANOMALY_CLASSES}')
    print(f'  ✓ CAT3_REMAP: {CAT3_REMAP}')

NUM_CLASSES2 = len(ANOMALY_CLASSES)  # background + N diseases
print(f'  NUM_CLASSES2 = {NUM_CLASSES2}  (background + {NUM_CLASSES2-1} disease classes)')

ANOMALY_COLORS = {name: _COLOR_POOL.get(name, (128,128,255))
                  for name in ANOMALY_CLASSES.values() if name != 'background'}
ANOMALY_COLORS['Healthy'] = (0, 220, 100)
print(f'  ANOMALY_COLORS: {ANOMALY_COLORS}')

# Count how many diseased annotations we actually have
_diseased_count = sum(1 for a in _anns if a.get('category_id_3', 0) > 0)
_healthy_count  = sum(1 for a in _anns if a.get('category_id_3', 0) == 0)
print(f'\n  Diseased annotations : {_diseased_count}')
print(f'  Healthy  annotations : {_healthy_count} (will be skipped in training)')

# ─ Dataset class ──────────────────────────────────────────────────────────────────
class DentexAnomalyDataset(Dataset):
    def __init__(self, img_dir, ann_file, image_ids=None, cat3_remap=None):
        self.img_dir     = img_dir
        self.cat3_remap  = cat3_remap or CAT3_REMAP
        with open(ann_file) as f:
            raw = json.load(f)
        imgs_list = raw.get('images', [])
        anns_list = raw.get('annotations', [])
        self.id_to_file = {img['id']: img['file_name'] for img in imgs_list}
        # Group annotations by image_id; only keep diseased (category_id_3 > 0)
        ann_map = {}
        for ann in anns_list:
            if ann.get('category_id_3', 0) > 0:  # skip healthy
                ann_map.setdefault(ann['image_id'], []).append(ann)
        if image_ids is None:
            image_ids = list(self.id_to_file.keys())
        self.samples = []
        skipped = 0
        for iid in image_ids:
            fname = self.id_to_file.get(iid, '')
            fpath = os.path.join(img_dir, fname)
            if not os.path.exists(fpath):
                skipped += 1; continue
            anns = ann_map.get(iid, [])
            if len(anns) == 0:
                skipped += 1; continue
            self.samples.append((iid, fname, anns))
        print(f'  ✓ DentexAnomalyDataset: {len(self.samples)} images with disease annotations  (skipped {skipped})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        iid, fname, anns = self.samples[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        img_np = np.array(img, dtype=np.float32) / 255.0
        boxes, labels = [], []
        for ann in anns:
            orig_cid = ann.get('category_id_3', 0)
            model_lbl = self.cat3_remap.get(orig_cid)
            if model_lbl is None: continue  # not in remap = ignore
            x, y, w, h = ann['bbox']
            x1, y1, x2, y2 = x, y, x+w, y+h
            if x2 <= x1 or y2 <= y1: continue
            boxes.append([x1, y1, x2, y2])
            labels.append(model_lbl)
        if len(boxes) == 0:
            boxes = [[0.,0.,1.,1.]]; labels = [0]
        boxes  = torch.as_tensor(boxes,  dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {
            'boxes':    boxes,
            'labels':   labels,
            'image_id': torch.tensor([iid]),
            'area':     (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1]),
            'iscrowd':  torch.zeros(len(labels), dtype=torch.int64),
        }
        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1)
        return img_tensor, target

del _raw, _images, _anns, _cats3
print('[CELL 2] ✓ Dataset class defined. ANOMALY_CLASSES and NUM_CLASSES2 updated.')

In [ ]:
# ================== CELL 3: SPLIT & LOADERS ==================
with open(DENTEX_ANN_FILE) as f:
    _meta = json.load(f)
all_ids = [img['id'] for img in _meta.get('images', [])]
del _meta

random.seed(42); random.shuffle(all_ids)
n_total = len(all_ids)
n_train = int(n_total * CONFIG2['train_ratio'])
n_val   = int(n_total * CONFIG2['val_ratio'])
train_ids = all_ids[:n_train]
val_ids   = all_ids[n_train:n_train+n_val]
test_ids  = all_ids[n_train+n_val:]
print(f'  Total: {n_total}  | Train: {len(train_ids)}  | Val: {len(val_ids)}  | Test: {len(test_ids)}')

train_ds = DentexAnomalyDataset(DENTEX_IMG_DIR, DENTEX_ANN_FILE, image_ids=train_ids)
val_ds   = DentexAnomalyDataset(DENTEX_IMG_DIR, DENTEX_ANN_FILE, image_ids=val_ids)
test_ds  = DentexAnomalyDataset(DENTEX_IMG_DIR, DENTEX_ANN_FILE, image_ids=test_ids)

if len(train_ds) == 0:
    raise ValueError('❌ Train dataset empty! No images have disease annotations in train split.')

def collate_fn(batch): return tuple(zip(*batch))

train_loader = DataLoader(train_ds, batch_size=CONFIG2['batch_size'], shuffle=True,
                          collate_fn=collate_fn, num_workers=CONFIG2['num_workers'])
val_loader   = DataLoader(val_ds,   batch_size=CONFIG2['batch_size'], shuffle=False,
                          collate_fn=collate_fn, num_workers=CONFIG2['num_workers'])
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          collate_fn=collate_fn, num_workers=CONFIG2['num_workers'])
print(f'  ✓ Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# ================== CELL 4: FASTER R-CNN MODEL ==================
def get_anomaly_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model2 = get_anomaly_model(NUM_CLASSES2)
model2.to(DEVICE)
optimizer2 = torch.optim.AdamW([p for p in model2.parameters() if p.requires_grad],
                               lr=CONFIG2['lr'], weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=CONFIG2['epochs'], eta_min=1e-6)
print(f'[CELL 4] ✓ Faster R-CNN on {DEVICE}')
print(f'  Output classes: {NUM_CLASSES2}  → {list(ANOMALY_CLASSES.values())}')

In [ ]:
# ================== CELL 5: TRAINING LOOP ==================
best_val_loss2 = float('inf'); patience_counter = 0
train_losses2, val_losses2 = [], []

for epoch in range(CONFIG2['epochs']):
    model2.train()
    epoch_train = 0.0
    for imgs, tgts in train_loader:
        imgs = [i.to(DEVICE) for i in imgs]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
        loss_dict = model2(imgs, tgts)
        loss = sum(loss_dict.values())
        optimizer2.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        optimizer2.step()
        epoch_train += loss.item()
    avg_train = epoch_train / max(1, len(train_loader))
    train_losses2.append(avg_train)

    model2.train()
    epoch_val = 0.0
    with torch.no_grad():
        for imgs, tgts in val_loader:
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            epoch_val += sum(model2(imgs, tgts).values()).item()
    avg_val = epoch_val / max(1, len(val_loader))
    val_losses2.append(avg_val)
    scheduler2.step()

    print(f'Epoch {epoch+1:03d}/{CONFIG2["epochs"]} | Train: {avg_train:.4f} | Val: {avg_val:.4f}', end='')
    if avg_val < best_val_loss2:
        best_val_loss2 = avg_val; patience_counter = 0
        torch.save({'epoch': epoch+1, 'model_state_dict': model2.state_dict(),
                    'optimizer_state_dict': optimizer2.state_dict(),
                    'val_loss': best_val_loss2,
                    'anomaly_classes': ANOMALY_CLASSES,
                    'cat3_remap': CAT3_REMAP},
                   '/kaggle/working/stage2_anomaly_best.pth')
        print('  ← best ✓', end='')
    else:
        patience_counter += 1
    print()
    if patience_counter >= CONFIG2['patience']:
        print(f'  Early stopping at epoch {epoch+1}'); break

torch.save(model2.state_dict(), '/kaggle/working/stage2_anomaly_final.pth')
print('\n[CELL 5] ✓ Training complete.')
plt.figure(figsize=(10,4))
plt.plot(train_losses2, label='Train'); plt.plot(val_losses2, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Stage 2 Training Loss')
plt.legend(); plt.tight_layout()
plt.savefig('/kaggle/working/stage2_loss_curve.png', dpi=120); plt.show()

In [ ]:
# ================== CELL 6: LOAD MODELS & HELPERS ==================
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision.transforms.functional as TF

def get_stage1_model(num_classes=33):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    in_feat_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_feat_mask, 256, num_classes)
    return model

stage1 = get_stage1_model(33)
ckpt1  = torch.load(STAGE1_WEIGHTS, map_location=DEVICE)
stage1.load_state_dict(ckpt1.get('model_state_dict', ckpt1), strict=False)
stage1.to(DEVICE).eval()
print('[CELL 6] ✓ Stage 1 (Mask R-CNN) loaded')

stage2 = get_anomaly_model(NUM_CLASSES2)
ckpt2  = torch.load('/kaggle/working/stage2_anomaly_best.pth', map_location=DEVICE)
stage2.load_state_dict(ckpt2['model_state_dict'])
stage2.to(DEVICE).eval()
# Restore class map from checkpoint (in case this cell is run in a new session)
if 'anomaly_classes' in ckpt2:
    ANOMALY_CLASSES = ckpt2['anomaly_classes']
    print(f'  Restored ANOMALY_CLASSES from checkpoint: {ANOMALY_CLASSES}')
print('[CELL 6] ✓ Stage 2 (Faster R-CNN anomaly) loaded')

def infer_stage1(model, img_tensor, conf=0.5, iou_thr=0.3):
    with torch.no_grad():
        pred = model([img_tensor.to(DEVICE)])[0]
    keep = pred['scores'] >= conf
    pred = {k: v[keep] for k, v in pred.items()}
    if len(pred['boxes']):
        idx = nms(pred['boxes'], pred['scores'], iou_thr)
        pred = {k: v[idx] for k, v in pred.items()}
    return {k: v.cpu().numpy() for k in ['boxes','labels','masks','scores']}

def infer_stage2(model, img_tensor, conf=None, iou_thr=None):
    conf    = conf    or CONFIG2['conf_threshold']
    iou_thr = iou_thr or CONFIG2['nms_iou']
    with torch.no_grad():
        pred = model([img_tensor.to(DEVICE)])[0]
    keep = pred['scores'] >= conf
    pred = {k: v[keep] for k, v in pred.items()}
    if len(pred['boxes']):
        idx = nms(pred['boxes'], pred['scores'], iou_thr)
        pred = {k: v[idx] for k, v in pred.items()}
    return {k: v.cpu().numpy() for k in ['boxes','labels','scores']}

def assign_quadrant(cx, cy, img_w, img_h):
    row = 'U' if cy < img_h/2 else 'L'
    col = 'L' if cx < img_w/2 else 'R'
    return row+col

print('[CELL 6] ✓ All helpers defined.')

In [ ]:
# ================== CELL 7: PER-TOOTH PIPELINE ==================
VIBRANT = [(255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255),(0,255,255),
           (255,128,0),(128,0,255),(255,0,128),(0,255,128),(128,255,0),(0,128,255)]

def run_full_pipeline(img_path, pad=20):
    img_pil = Image.open(img_path).convert('RGB')
    img_np  = np.array(img_pil)
    H, W    = img_np.shape[:2]
    img_t   = TF.to_tensor(img_pil)

    s1 = infer_stage1(stage1, img_t)
    if len(s1['boxes']) == 0:
        print('  ⚠  Stage 1 found no teeth.'); return None

    x1c = max(0, int(s1['boxes'][:,0].min()) - pad)
    y1c = max(0, int(s1['boxes'][:,1].min()) - pad)
    x2c = min(W, int(s1['boxes'][:,2].max()) + pad)
    y2c = min(H, int(s1['boxes'][:,3].max()) + pad)
    cropped_np = img_np[y1c:y2c, x1c:x2c]

    colored_np = cropped_np.copy()
    for i, (mask, _) in enumerate(zip(s1['masks'], s1['labels'])):
        mb = (mask[0] > 0.5).astype(np.uint8)
        mb_crop = mb[y1c:y2c, x1c:x2c]
        if mb_crop.shape != colored_np.shape[:2]: continue
        color = VIBRANT[i % len(VIBRANT)]
        ov = colored_np.copy()
        for c in range(3): ov[:,:,c] = np.where(mb_crop==1, color[c], ov[:,:,c])
        colored_np = cv2.addWeighted(colored_np, 0.3, ov, 0.7, 0)
        cnts, _ = cv2.findContours(mb_crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(colored_np, cnts, -1, color, 2)

    buckets = {'UL':[], 'UR':[], 'LL':[], 'LR':[]}
    for i, (box, lbl, mask, sc) in enumerate(zip(s1['boxes'], s1['labels'], s1['masks'], s1['scores'])):
        bx1,by1,bx2,by2 = box
        tcx, tcy = int((bx1+bx2)/2), int((by1+by2)/2)
        buckets[assign_quadrant(tcx,tcy,W,H)].append(
            {'idx':i,'box':box,'mask':mask,'score':float(sc),'centroid':(tcx,tcy)})

    numbered_quads = {}
    for q, teeth in buckets.items():
        teeth = sorted(teeth, key=lambda t: -t['centroid'][0] if q in ('UL','LL') else t['centroid'][0])[:8]
        for n, t in enumerate(teeth, 1):
            t['number'] = n; t['quadrant'] = q
        numbered_quads[q] = teeth

    anomaly_map = {}
    for tooth in [t for teeth in numbered_quads.values() for t in teeth]:
        bx1,by1,bx2,by2 = [int(v) for v in tooth['box']]
        tx1=max(0,bx1-8); ty1=max(0,by1-8); tx2=min(W,bx2+8); ty2=min(H,by2+8)
        tooth_crop = img_pil.crop((tx1,ty1,tx2,ty2))
        if tooth_crop.width < 4 or tooth_crop.height < 4: continue
        s2 = infer_stage2(stage2, TF.to_tensor(tooth_crop))
        if len(s2['boxes']) > 0:
            tooth_anoms = []
            for lb, sc in zip(s2['labels'], s2['scores']):
                lname = ANOMALY_CLASSES.get(int(lb), f'cls_{lb}')
                if lname == 'background': continue
                tooth_anoms.append({'label':int(lb),'label_name':lname,'score':float(sc)})
            if tooth_anoms:
                anomaly_map[(tooth['quadrant'],tooth['number'])] = {'anomalies':tooth_anoms,'box':tooth['box']}

    print(f'  Stage 1: {len(s1["boxes"])} teeth | Stage 2: {len(anomaly_map)} anomalies')
    return img_np, cropped_np, colored_np, numbered_quads, anomaly_map, ((x1c+x2c)//2,(y1c+y2c)//2), (x1c,y1c)

print('[CELL 7] ✓ Full pipeline function defined.')

In [ ]:
# ================== CELL 8: VISUALIZATION ==================
def visualize_pipeline(img_path):
    result = run_full_pipeline(img_path)
    if result is None: return
    img_np, cropped_np, colored_np, numbered_quads, anomaly_map, (cx_img,cy_img), (ox,oy) = result

    annotated = colored_np.copy()
    ch, cw    = annotated.shape[:2]
    cx_crop, cy_crop = cx_img-ox, cy_img-oy
    cv2.line(annotated, (cx_crop,0), (cx_crop,ch), (255,255,0), 3)
    cv2.line(annotated, (0,cy_crop), (cw,cy_crop), (255,255,0), 3)

    for q_name, teeth in numbered_quads.items():
        for tooth in teeth:
            tcx = tooth['centroid'][0]-ox; tcy = tooth['centroid'][1]-oy
            if not (0<=tcx<cw and 0<=tcy<ch): continue
            key = (q_name, tooth['number'])
            is_anomaly = key in anomaly_map
            if is_anomaly:
                bx1,by1,bx2,by2 = [int(v) for v in tooth['box']]
                anom_name = anomaly_map[key]['anomalies'][0]['label_name']
                anom_color = ANOMALY_COLORS.get(anom_name, (255,50,50))
                cv2.rectangle(annotated,(bx1-ox,by1-oy),(bx2-ox,by2-oy),anom_color,4)
            num_str   = str(tooth['number'])
            txt_color = (255,80,80) if is_anomaly else (0,255,255)
            cv2.putText(annotated,num_str,(tcx-15,tcy+12),cv2.FONT_HERSHEY_SIMPLEX,1.2,(0,0,0),6)
            cv2.putText(annotated,num_str,(tcx-15,tcy+12),cv2.FONT_HERSHEY_SIMPLEX,1.2,txt_color,2)

    for label, pos in [('UL',(20,50)),('UR',(cw-70,50)),('LL',(20,ch-20)),('LR',(cw-70,ch-20))]:
        cv2.putText(annotated,label,pos,cv2.FONT_HERSHEY_SIMPLEX,1.8,(0,0,0),6)
        cv2.putText(annotated,label,pos,cv2.FONT_HERSHEY_SIMPLEX,1.8,(0,255,255),3)

    fig, axes = plt.subplots(2,2,figsize=(22,16))
    fig.patch.set_facecolor('#111')
    for ax in axes.flat: ax.set_facecolor('#111')
    axes[0,0].imshow(img_np);     axes[0,0].set_title('Original Panoramic X-ray',         color='w',fontsize=15,fontweight='bold')
    axes[0,1].imshow(cropped_np); axes[0,1].set_title('Cropped to Teeth Region',           color='w',fontsize=15,fontweight='bold')
    axes[1,0].imshow(colored_np); axes[1,0].set_title('Segmented & Colored Teeth',         color='w',fontsize=15,fontweight='bold')
    axes[1,1].imshow(annotated);  axes[1,1].set_title('Quadrants + Numbering + Anomalies', color='w',fontsize=15,fontweight='bold')
    for ax in axes.flat: ax.axis('off')
    patches = [mpatches.Patch(color=np.array(v)/255, label=k) for k,v in ANOMALY_COLORS.items()]
    fig.legend(handles=patches,loc='lower center',ncol=5,fontsize=12,
               facecolor='#222',labelcolor='white',framealpha=0.8)
    plt.tight_layout(rect=[0,0.05,1,1])
    plt.savefig('/kaggle/working/pipeline_result_'+os.path.basename(img_path).replace(' ','_')+'.png',
                dpi=150,bbox_inches='tight')
    plt.show()

    print('\n' + '='*70)
    print('CLINICAL ANOMALY REPORT')
    print('='*70)
    all_teeth = [t for teeth in numbered_quads.values() for t in teeth]
    for tooth in sorted(all_teeth, key=lambda t:(t['quadrant'],t['number'])):
        key = (tooth['quadrant'],tooth['number'])
        if key not in anomaly_map:
            print(f'  Tooth {tooth["number"]:>2}  [{tooth["quadrant"]}]  →  Healthy ✓')
        else:
            for a in anomaly_map[key]['anomalies']:
                print(f'  Tooth {tooth["number"]:>2}  [{tooth["quadrant"]}]  →  ⚠  {a["label_name"]}  [{a["score"]*100:.0f}%]')
    print('='*70)

print('[CELL 8] ✓ Visualization function defined.')

In [ ]:
# ================== CELL 9: RUN INFERENCE ==================
import glob
xray_files = sorted(
    glob.glob(os.path.join(INFER_IMG_DIR,'*.png')) +
    glob.glob(os.path.join(INFER_IMG_DIR,'*.jpg'))
)
print(f'[CELL 9] Found {len(xray_files)} panoramic X-rays')
for i, fpath in enumerate(xray_files[:3]):
    print(f'\n{"="*60}\nSAMPLE {i+1}: {os.path.basename(fpath)}\n{"="*60}')
    visualize_pipeline(fpath)